In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from evaluate import (
    SCORE_COLS, SECTION_LABELS,
    discover_runs, load_results, truth_coverage,
    run_evaluations, load_eval_csv,
    report_latency, report_scores, browse_responses,
    plot_section_heatmap, plot_s4_accuracy, plot_latency, plot_histogram, plot_comparison,
    short_name,
)
print('evaluate loaded OK')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT_ROOT      = Path('results')
TRUTH_ROOT       = Path('truth')
EVAL_OUTPUT_ROOT = Path('eval_output')
EVAL_CSV         = EVAL_OUTPUT_ROOT / 'eval_scores.csv'
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## i. Discover available results
Scan `results/` and show what runs are available, what models were used, and which context backend each run used. Use this to decide which runs and models to load and compare.

In [ ]:
# ── Overview: all runs ────────────────────────────────────────────────────────
runs_df = discover_runs(OUTPUT_ROOT)
display(runs_df)

In [ ]:
# ── Per-run detail: read config JSON for full BatchConfig ─────────────────────
import json

RUN_IDS_INSPECT = list(runs_df['run_id'])   # or narrow to specific run_ids

for run_id in RUN_IDS_INSPECT:
    cfg_files = list((OUTPUT_ROOT / run_id).glob('config_*.json'))
    if not cfg_files:
        print(f'{run_id}: no config JSON')
        continue
    cfg = json.loads(cfg_files[0].read_text())
    print(f"{'─'*60}")
    print(f"run_id      : {cfg.get('run_id')}")
    print(f"context     : {cfg.get('context', {}).get('type')}  "
          f"{cfg.get('context', {})}")
    print(f"models      : {[m['name'] if isinstance(m, dict) else m for m in cfg.get('models', [])]}")
    print(f"plot_filter : {cfg.get('plot_filter')}")
    print(f"ref_dir     : {cfg.get('ref_dir')}")
    print(f"run_metadata: {cfg.get('run_metadata')}")

## ii. Load and summarize
Select run IDs to load, then inspect response counts, model coverage, and latency. No judge call needed for this section.

In [ ]:
# ── Select run IDs and load ───────────────────────────────────────────────────
RUN_IDS = ['localRAG','YAML']   # pick from discover output above
MODELS  = None                    # None = all models found; or ['google/gemma4-31b', ...]

df = load_results(OUTPUT_ROOT, RUN_IDS, models=MODELS)

In [ ]:
df.plot_name.unique()

In [ ]:
# ── Coverage: run_id × model × plot ──────────────────────────────────────────
print('=== Response count by run_id × model ===')
display(
    df.groupby(['run_id', 'model_short'])['image_name']
    .count().rename('n_responses')
    .unstack('model_short')
    .fillna(0).astype(int)
)

print('\n=== Plots covered per run_id ===')
display(
    df.groupby('model_short')['plot_name']
    .apply(lambda s: ', '.join(sorted(s.unique())))
    .rename('plots')
)

In [ ]:
# ── Truth coverage: which images have ground-truth files? ─────────────────────
cov = truth_coverage(df, TRUTH_ROOT)
display(cov)

In [ ]:
# ── Latency report (from raw responses, no judge needed) ──────────────────────
report_latency(df, group_col='model_short')

In [ ]:
# ── Error check ───────────────────────────────────────────────────────────────
errors = df[df['error'].notna()]
if errors.empty:
    print('No errors.')
else:
    print(f'{len(errors)} errors:')
    display(errors[['run_id', 'model', 'plot_name', 'image_name', 'error']])

## iii. Configure and run evaluation
Choose a judge model and which models to evaluate. Results are cached in `EVAL_CSV` — interrupted runs resume automatically.

In [ ]:
# ── Judge configuration ───────────────────────────────────────────────────────
# Judge model: should be different from the evaluated models.
# Use list_models() from owui_client to check availability.
JUDGE_MODEL = 'openai/gpt-oss-120b'
DELAY       = 1.0   # seconds between judge calls

# Map each evaluated model to one or more judge models.
# Models not in this dict are skipped.
JUDGE_MODEL_FOR = {m: [JUDGE_MODEL] for m in df['model'].unique()}
print('Judge config:')
for m, js in JUDGE_MODEL_FOR.items():
    print(f'  {short_name(m):20s} → {js}')

In [ ]:
# ── Run evaluations (cached) ──────────────────────────────────────────────────
# Results are appended to EVAL_CSV row-by-row as they complete.
# Reload from EVAL_CSV at any time with load_eval_csv() — no need to re-judge.
df_eval = run_evaluations(
    df, TRUTH_ROOT, JUDGE_MODEL_FOR, EVAL_CSV, delay=DELAY,
)
print(f'{len(df_eval)} scored rows in df_eval')

In [ ]:
# ── Reload from CSV (skip judge loop) ────────────────────────────────────────
# Run this cell instead of the one above to load previously saved scores.
df_eval = load_eval_csv(EVAL_CSV)

In [ ]:
df_eval

In [ ]:
# ── Score report ──────────────────────────────────────────────────────────────
# PRIMARY_AXIS controls how the report groups results:
#   'model'  → compare models within each run_id
#   'run_id' → compare context backends within each model
PRIMARY_AXIS = 'model_short'

report_scores(df_eval, group_col=PRIMARY_AXIS)

## iv. Comparison plots
`df_eval` carries two derived columns beyond the raw judge output: **`run_number`** (the numeric
DQM run parsed from the image filename) and **`subsystem`** (the prefix of `plot_name`, e.g. `L1T`).

Every plot function below takes the same filter controls — `run_id`, `plot_name`, `model` — where
`None` (default) includes everything, a single value is an exact match, and a list restricts to
those values. `run_number` is never filtered; it's always reduced via `AGG`: `'mean'` (default),
`'median'`, or `'std'`.

`out_path` is optional — pass a `Path` to also save a PNG, or omit it to just preview inline.


In [ ]:
# ── Possible values (copy/paste into the cells below) ─────────────────────────
print('model       :', sorted(df_eval['model'].dropna().unique()))
print('model_short :', sorted(df_eval['model_short'].dropna().unique()))
print('run_id      :', sorted(df_eval['run_id'].dropna().unique()))
print('plot_name   :', sorted(df_eval['plot_name'].dropna().unique()))
print('run_number  :', sorted(df_eval['run_number'].dropna().unique()))


## Overall section score

In [ ]:
COMPARE   = 'model'    # dimension to compare: run_number, run_id, model, model_short
models    =  None     # exact value, list or None(default) — filters the 'model' column
plot_name =  None      # exact value, list or None(default)
run_id    =  "YAML"     # exact value, list or None(default)
AGG       = 'mean'      # how to reduce over run_number: 'mean', 'median', or 'std'

ax = plot_section_heatmap(
    df_eval, COMPARE, model=models, run_id=run_id,plot_name=plot_name, 
    title=f"Mean score per section for all models",
    # out_path=EVAL_OUTPUT_ROOT / 'examples' / 'heatmap_test.png',
)


## Overall section score — one subplot per model
`plot_section_heatmap`'s `subplot_col` draws one heatmap panel per `subplot_col` value
(e.g. one per model), stacked as separate Axes in a single figure — each panel's rows are
still `row_col` (e.g. `run_id`).


In [ ]:
COMPARE     = 'run_id'                                                    # heatmap rows within each subplot
SUBPLOT     = 'model_short'                                               # one subplot per value of this dimension
models      =  ['google/gemma3-27b', 'google/gemma4-31b', 'qwen/qwen3.6']  # exact value, list or None(default) — filters the 'model' column
plot_name   =  None      # exact value, list or None(default)
run_id      =  None      # exact value, list or None(default) — leave None to compare across all run_ids
AGG         = 'mean'      # how to reduce over run_number: 'mean', 'median', or 'std'

axes = plot_section_heatmap(
    df_eval, COMPARE, model=models, run_id=run_id, plot_name=plot_name, agg=AGG,
    subplot_col=SUBPLOT,
    title=f"Mean score per section",
)


## Overall section score (list filtered)

In [ ]:
COMPARE   = 'model_short'    # dimension to compare: run_number, run_id, model, model_short
models    =  sorted(df_eval['model'].dropna().unique())[:2]   # exact value, list or None(default) — filters the 'model' column
plot_name =  None       # exact value, list or None(default)
run_id    =  "YAML"     # exact value, list or None(default)
AGG       = 'mean'      # how to reduce over run_number: 'mean', 'median', or 'std'

ax = plot_section_heatmap(
    df_eval, COMPARE, model=models, run_id=run_id, plot_name=plot_name, agg=AGG,
    title=f"Mean score per section  |  model in {models}",
)


## Overall section score vs run number

In [ ]:
[x for x in df_eval['model'].dropna().unique() if "claude" in x]

In [ ]:
COMPARE   = 'run_number'    # dimension to compare: run_number, run_id, model, model_short
SUBPLOT   = 'model_short'   
models    =  [x for x in df_eval['model'].dropna().unique() if "claude" in x]      # exact value, list or None(default) — filters the 'model' column
plot_name =  None      # exact value, list or None(default)
run_id    =  "YAML"     # exact value, list or None(default)
AGG       = 'mean'      # how to reduce over run_number: 'mean', 'median', or 'std'

ax = plot_section_heatmap(
    df_eval, COMPARE, model=models, run_id=run_id, plot_name=plot_name, agg=AGG,subplot_col=SUBPLOT,
    title=f"Mean score per section vs {COMPARE}",
)


## Overall latency comparison
Same standard-cell style, but `plot_latency` with `overlay=True` draws every `COMPARE` value as
outlined step histograms on one shared plot — e.g. all models overlaid — instead of stacking a
separate subplot per value (`overlay=False`).


In [ ]:
COMPARE   = 'model_short'    # dimension to compare: run_number, run_id, model, model_short
models    =  [x for x in df_eval['model_short'].dropna().unique() if "gemma" in x]      # exact value, list or None(default) — filters the 'model' column
plot_name =  None      # exact value, list or None(default)
run_id    =  "YAML"     # exact value, list or None(default)
AGG       = 'mean'      # reference line drawn per value: 'mean', 'median', or 'std'

fig, ax = plot_latency(
    df_eval, COMPARE, model=models, run_id=run_id, plot_name=plot_name, agg=AGG,
    overlay=True,
    title=f"Generation latency by {COMPARE}",
)


## v. Browse individual responses
Useful for diagnosing specific failures after looking at the aggregate comparison plots above.
`browse_responses` joins raw responses (`df`) with ground-truth text (`TRUTH_ROOT`) and, if
`df_eval` is passed, judge scores/comments — and prints the result as a table.

`run_id`, `plot_name`, `model`, `image_name`, `judge_model` — same filter convention as the plot
functions above: `None` (default) includes everything, a single value is an exact match, and a
list restricts to those values. `n` caps the number of rows shown (`None` = no cap).


In [ ]:
# ── Possible values (copy/paste into the cells below) ─────────────────────────
print('model       :', sorted(df_eval['model'].dropna().unique()))
print('model_short :', sorted(df_eval['model_short'].dropna().unique()))
print('run_id      :', sorted(df_eval['run_id'].dropna().unique()))
print('plot_name   :', sorted(df_eval['plot_name'].dropna().unique()))
print('run_number  :', sorted(df_eval['run_number'].dropna().unique()))


In [ ]:
run_id      = None      # exact value, list or None(default)
plot_name   = None      # exact value, list or None(default)
model       = ['gemma4-31b']      # exact value, list or None(default) — accepts full or short model names
image_name  = None      # exact value, list or None(default)
judge_model = None      # exact value, list or None(default) — only applies when df_eval is passed
run_number  = 398185     # exact value, list or None(default)
N           = 5         # cap the number of rows shown; None = no cap

_ = browse_responses(
    df, df_eval, TRUTH_ROOT,
    run_id=run_id, plot_name=plot_name, model=model,
    image_name=image_name, judge_model=judge_model, run_number=run_number, n=N,
)
